# LakePilot Revision — Phase 7 training (Kaggle GPU)

Trains two independent things. **Neither changes the three-architecture comparison
from Phase 1**, which stays as it is: all three architectures trained under
identical configuration (same pools, 150/200 epochs, batch 64, balanced sampling,
5 seeds) and evaluated on matched environment seeds.

### 1. Parameter-matched MLP  (`M`, 5 seeds)
Controls the capacity confound the reviewer raised. The published models differ in
both encoder *and* size, in **opposite directions per specialist**:

| specialist | AttentivePPO | MLP-PPO |
|---|---|---|
| compaction | 84,741 | 70,469 (**17 % fewer**) |
| partition | 85,830 | 121,734 (**42 % more**) |

Matched widths give 84,837 (+0.11 %) and 85,830 (+0.00 %).

### 2. Attention tuning study  (`A`–`E`, 3 seeds each)
A **separate, exploratory** section asking *"could attention do better with a
configuration suited to it?"* — not entered into the three-architecture
comparison, because that comparison is defined by matched configuration.

Two honesty rules apply to how these are reported:
1. **Every variant trained is reported**, not only the best — otherwise it is
   selection on noise.
2. **DDQN was not given an equivalent sweep.** "Tuned attention ≈ DDQN" would mean
   *attention can be brought closer with tuning*, not *attention is as good*.

The most interesting knob is **window size**: attention's premise is modelling
relationships across a sequence, yet the compaction specialist sees only 10 steps
before global average pooling — little for attention to exploit over a flat MLP.

**How to run**: attach the same `kaggle_training_data_v2` dataset used for Phase 1,
enable GPU, Run All (~1 h). Download `phase7_weights.zip`.


In [ ]:
# ── Cell 1: setup ──
import os, glob, json, time, zipfile, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf

print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))
try:
    GPU_NAME = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                                       text=True).strip().splitlines()[0]
except Exception:
    GPU_NAME = 'none'
print("GPU:", GPU_NAME)

OUT = Path('/kaggle/working/phase7_weights') if Path('/kaggle').exists() else Path('./phase7_weights')
(OUT/'weights').mkdir(parents=True, exist_ok=True)
(OUT/'curves').mkdir(parents=True, exist_ok=True)

# Constants — identical to Phase 1 (revision/phase1_stats/kaggle_train_phase1.ipynb)
ROWS_MEAN, ROWS_STD = 94.0, 71.0
RATE_MEAN, RATE_STD = 318.0, 218.0
LATENCY_MAX, FILE_COUNT_MAX = 15000.0, 2000.0
COMPACT_COST_MAX, PARTITION_COST_MAX, SKEW_MAX = 1.1, 2.0, 2.0
CW_FILES, CW_UTIL, CW_COST, CW_TARGET = 0.25, 0.20, 0.10, 0.45
PW_PRUNING, PW_DELTA, PW_SKEW, PW_COST = 0.55, 0.25, 0.10, 0.10
GW_LATENCY, GW_FILES, GW_PRUNING, GW_UTIL = 0.30, 0.20, 0.30, 0.20
ACTION_COSTS = {0:0.0, 1:0.9, 2:1.0, 3:1.1, 4:2.0, 5:2.0, 6:2.0, 7:0.7}
GAMMA, GAMMA_COMPACT = 0.99, 0.95

def set_seeds(seed):
    import random as _r
    _r.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)


In [ ]:
# ── Cell 2: data (identical pools to Phase 1) ──
DATA_GLOBS = ['/kaggle/input/**/*_ep*.csv', './kaggle_training_data/**/*_ep*.csv']
files = []
for p in DATA_GLOBS:
    files.extend(glob.glob(p, recursive=True))
EXCLUDE = ('eval_', 'MultiAgent', 'CompactOnly', 'PartitionOnly')
files = sorted(set(f for f in files
                   if 'transitions' in Path(f).name and not any(x in f for x in EXCLUDE)))
assert files, "attach the kaggle_training_data_v2 dataset"

PARTITION_SOURCES = ('WorkloadAwareThreshold', 'PartitionExploration', 'Random_Partition')
files_p = [f for f in files if any(s in f for s in PARTITION_SOURCES)]
files_c = [f for f in files if f not in files_p]
n_rec = sum('RecoveryExploration' in f for f in files_c)
print(f"compaction pool {len(files_c)} files ({n_rec} recovery) | partition pool {len(files_p)}")
assert len(files_c) - n_rec == 14 and len(files_p) == 6 and n_rec >= 3

def load_pool(pool):
    fr = []
    for i, f in enumerate(pool):
        d = pd.read_csv(f); d['ep'] = i; fr.append(d)
    return pd.concat(fr, ignore_index=True).sort_values(['ep','step']).reset_index(drop=True)

def target_bonus_vec(a, rows):
    tb = np.zeros(len(a), dtype=np.float32)
    low, mid, high = rows <= 50, (rows > 50) & (rows < 150), rows >= 150
    for act, l, m, h in [(1,1.0,0.15,-0.6), (2,0.25,1.0,0.4), (3,-0.5,0.15,1.0)]:
        s = a == act
        tb[s & low], tb[s & mid], tb[s & high] = l, m, h
    return tb

def recompute_rewards(d):
    d = d.copy()
    a = d['action'].fillna(0).astype(int).values
    cost = np.vectorize(lambda x: ACTION_COSTS.get(x, 0.0))(a)
    fc = d.get('next_file_count', d.get('file_count')).astype(float).values
    util = d.get('next_block_utilization', d.get('block_utilization')).astype(float).values
    lat = d.get('next_latency_ms', d.get('latency_ms')).astype(float).values
    rows = d.get('rows_ingested', pd.Series(np.zeros(len(d)))).astype(float).values
    skew = d.get('file_size_skew_kb', pd.Series(np.zeros(len(d)))).astype(float).values
    pr_c = d.get('partition_pruning_ratio', pd.Series(np.zeros(len(d)))).astype(float).values
    pr_n = d.get('next_partition_pruning_ratio', pd.Series(pr_c)).astype(float).values
    f_n = np.clip(fc/FILE_COUNT_MAX,0,1); u_n = np.clip(util,0,1)
    l_n = np.clip(lat/LATENCY_MAX,0,1);   s_n = np.clip(skew/SKEW_MAX,0,1)
    p_n = np.clip(pr_n,0,1);              dp = np.clip(pr_n-pr_c,-1,1)
    d['compact_reward'] = (-CW_FILES*f_n + CW_UTIL*u_n - CW_COST*cost/COMPACT_COST_MAX
                           + CW_TARGET*target_bonus_vec(a, rows))
    d['global_reward'] = -GW_LATENCY*l_n - GW_FILES*f_n + GW_PRUNING*p_n + GW_UTIL*u_n
    return d

df_c = recompute_rewards(load_pool(files_c))
df_pf = recompute_rewards(load_pool(files_p))
print(f"compaction rows {len(df_c)} | partition rows {len(df_pf)}")


In [ ]:
# ── Cell 3: features + windows (window size is a VARIABLE here) ──
def compact_features(d):
    f = np.zeros((len(d), 8), dtype=np.float32)
    f[:,0] = (d['rows_ingested'].values - ROWS_MEAN)/ROWS_STD
    f[:,1] = (d['ingestion_rate_rows_per_sec'].values - RATE_MEAN)/RATE_STD
    f[:,2] = np.clip(d['latency_ms'].values/15000.,0,1)
    f[:,3] = np.clip(d['file_count'].values/2000.,0,1)
    f[:,4] = np.clip(d['block_utilization'].values,0,1)
    f[:,5] = np.clip(d['total_size_kb'].values/50000.,0,1)
    f[:,6] = np.clip(d['file_size_skew_kb'].values/50.,0,1)
    f[:,7] = np.exp(-np.minimum(d['steps_since_compact'].values,10)/10.)
    return f

def partition_features(d):
    f = np.zeros((len(d), 14), dtype=np.float32)
    f[:,0] = np.clip(d['latency_ms'].values/15000.,0,1)
    f[:,1] = np.clip(d['file_count'].values/2000.,0,1)
    ps = d['partition_strategy'].values.astype(int)
    for i in range(4): f[:,2+i] = (ps == i).astype(np.float32)
    f[:,6] = np.clip(d['partition_pruning_ratio'].values,0,1)
    f[:,7] = np.clip(d['avg_pruning_ratio'].values,0,1)
    for j,c in enumerate(['query_hist_time_range','query_hist_region_filter',
                          'query_hist_sensor_lookup','query_hist_type_filter',
                          'query_hist_full_scan']):
        f[:,8+j] = d[c].values
    f[:,13] = np.exp(-np.minimum(d['steps_since_partition_change'].values,60)/30.)
    return f

def build_windows(F, ep_ids, window):
    out = np.zeros((len(F), window, F.shape[1]), dtype=np.float32)
    for e in np.unique(ep_ids):
        idx = np.where(ep_ids == e)[0]
        for j, i in enumerate(idx):
            s = max(0, j-window+1)
            seq = F[idx[s:j+1]]
            out[i, window-len(seq):, :] = seq
    return out

F_c, ep_c = compact_features(df_c), df_c['ep'].values
F_p, ep_p = partition_features(df_pf), df_pf['ep'].values

rew_c_full = df_c['compact_reward'].values.astype(np.float32)
G = np.zeros(len(df_c), dtype=np.float32)
for e in np.unique(ep_c):
    idx = np.where(ep_c == e)[0]; g = 0.0
    for i in idx[::-1]:
        g = rew_c_full[i] + GAMMA_COMPACT*g; G[i] = g
mask_c = df_c['action'].isin([0,1,2,3]).values
Ac, Gc = df_c['action'].values[mask_c].astype(np.int32), G[mask_c]

PA_MAP = {0:0, 4:1, 5:2, 6:3, 7:4}
mask_p = df_pf['action'].isin(PA_MAP).values
df_p = df_pf[mask_p].reset_index(drop=True)
Ap = np.array([PA_MAP[a] for a in df_p['action'].values], dtype=np.int32)

# partition shaped reward (published recipe)
lat_b = df_p['latency_ms'].values.astype(np.float32); lat_a = df_p['next_latency_ms'].values.astype(np.float32)
li = np.clip((lat_b-lat_a)/(lat_b+1.0), -1, 1)
pr_b = df_p['partition_pruning_ratio'].values.astype(np.float32)
pr_a = df_p['next_partition_pruning_ratio'].values.astype(np.float32)
pi_ = np.clip(pr_a-pr_b, -1, 1)
qh = np.column_stack([df_p['query_hist_time_range'], df_p['query_hist_region_filter'],
                      df_p['query_hist_sensor_lookup'], df_p['query_hist_type_filter']]).astype(np.float32)
dq, ds = np.argmax(qh, axis=-1), np.max(qh, axis=-1)
ideal = np.zeros(len(Ap), dtype=np.int32); ideal[dq==0], ideal[dq==1], ideal[dq==3] = 1, 2, 3
cps = df_p['partition_strategy'].values.astype(int)
okm = (cps == ideal) & (ideal > 0)
al = np.zeros(len(Ap), dtype=np.float32)
al[okm & (Ap==0)] = 0.5; al[okm & (Ap==4)] = -0.8
for a in [1,2,3]: al[okm & (Ap==a) & (Ap!=ideal)] = -0.5
bad = ~okm
for a in [1,2,3]: al[bad & (Ap==a) & (ideal==a)] = 1.0
al[bad & (Ap==0)] = -0.3; al[bad & (Ap==4) & (cps>0)] = 0.3
al *= np.clip(ds*2.5-0.5, 0.3, 1.5)
custom = (0.3*(0.5*li+0.5*pi_) + 0.5*al
          + 0.2*df_p['global_reward'].values.astype(np.float32)).astype(np.float32)

_WCACHE = {}
def windows_for(spec, window):
    key = (spec, window)
    if key not in _WCACHE:
        if spec == 'compaction':
            _WCACHE[key] = build_windows(F_c, ep_c, window)[mask_c]
        else:
            _WCACHE[key] = build_windows(F_p, ep_p, window)[mask_p]
    return _WCACHE[key]

def standardize(values, A, n_act):
    adv = np.asarray(values, dtype=np.float32).copy()
    for k in range(n_act):
        m = A == k
        if m.sum() > 1: adv[m] = (adv[m]-adv[m].mean())/(adv[m].std()+1e-8)
    return adv.astype(np.float32)

ADV_C, ADV_P = standardize(Gc, Ac, 4), standardize(custom, Ap, 5)
print(f"compaction {mask_c.sum()} | partition {mask_p.sum()}")


In [ ]:
# ── Cell 4: models (exact copies of src/agents/model_registry.py) ──
class AttentivePPOModel(tf.keras.Model):
    def __init__(self, num_actions, num_features, window_size=10, embed_dim=64, num_heads=4):
        super().__init__()
        self.num_actions = num_actions
        self.embedding = tf.keras.layers.Dense(embed_dim, activation='relu', name='embedding')
        self.pos_encoding = self.add_weight(name='pos_encoding', shape=(1, window_size, embed_dim),
                                            initializer='glorot_uniform', trainable=True)
        self.attention = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim,
                                                            dropout=0.1, name='self_attention')
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name='norm1')
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name='norm2')
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(embed_dim*2, activation='relu', name='ffn_dense1'),
            tf.keras.layers.Dropout(0.1),
            tf.keras.layers.Dense(embed_dim, name='ffn_dense2')], name='ffn')
        self.gap = tf.keras.layers.GlobalAveragePooling1D(name='gap')
        self.actor = tf.keras.layers.Dense(num_actions, activation='softmax', name='actor')
        self.critic = tf.keras.layers.Dense(1, name='critic')
    def call(self, x, training=False):
        e = self.embedding(x) + self.pos_encoding
        x1 = self.norm1(e + self.attention(e, e, training=training))
        x2 = self.norm2(x1 + self.ffn(x1, training=training))
        c = self.gap(x2)
        return self.actor(c), self.critic(c)

class MlpPPOModel(tf.keras.Model):
    def __init__(self, num_actions, num_features, window_size=10,
                 embed_dim=64, num_heads=4, hidden1=256, hidden2=128):
        super().__init__()
        self.num_actions = num_actions
        self.flatten = tf.keras.layers.Flatten()
        self.shared_fc1 = tf.keras.layers.Dense(hidden1, activation='relu', name='shared_fc1')
        self.shared_fc2 = tf.keras.layers.Dense(hidden2, activation='relu', name='shared_fc2')
        self.shared_dropout = tf.keras.layers.Dropout(0.1)
        self.actor_fc = tf.keras.layers.Dense(64, activation='relu', name='actor_fc')
        self.actor_out = tf.keras.layers.Dense(num_actions, activation='softmax', name='actor_out')
        self.critic_fc = tf.keras.layers.Dense(64, activation='relu', name='critic_fc')
        self.critic_out = tf.keras.layers.Dense(1, name='critic_out')
    def call(self, x, training=False):
        h = self.shared_dropout(self.shared_fc2(self.shared_fc1(self.flatten(x))), training=training)
        return self.actor_out(self.actor_fc(h)), self.critic_out(self.critic_fc(h))

MATCHED_MLP_HIDDEN = {(4,8,10): (128,288), (5,14,20): (128,192)}

def nparams(m): return int(sum(np.prod(v.shape) for v in m.trainable_variables))


In [ ]:
# ── Cell 5: PPO-clip trainer (identical objective to Phase 1's mlp/attention-clip) ──
def balanced_indices(A, n_act):
    per = {k: np.where(A == k)[0] for k in range(n_act)}
    spa = min(len(v) for v in per.values())
    idx = np.concatenate([np.random.choice(per[k], spa, replace=True) for k in range(n_act)])
    np.random.shuffle(idx)
    return idx

def train_ppoclip(model, X, A, ADV, R, n_act, epochs, lr=3e-4, batch=64,
                  clip_ratio=0.2, value_coef=0.5, ent=0.01):
    opt = tf.keras.optimizers.Adam(lr)
    init_probs, _ = model(tf.constant(X), training=False)
    old_logp = tf.math.log(tf.reduce_sum(init_probs*tf.one_hot(A, n_act), axis=-1)+1e-8).numpy()
    hist = []
    @tf.function
    def step(xb, ab, advb, rb, olb):
        with tf.GradientTape() as tape:
            probs, values = model(xb, training=True)
            values = tf.squeeze(values, -1)
            chosen = tf.reduce_sum(probs*tf.one_hot(ab, n_act), axis=-1)
            ratio = tf.exp(tf.math.log(chosen+1e-8) - olb)
            actor = -tf.reduce_mean(tf.minimum(ratio*advb,
                        tf.clip_by_value(ratio, 1-clip_ratio, 1+clip_ratio)*advb))
            critic = tf.reduce_mean(tf.square(rb - values))
            entropy = -tf.reduce_mean(tf.reduce_sum(probs*tf.math.log(probs+1e-8), -1))
            loss = actor + value_coef*critic - ent*entropy
        g = tape.gradient(loss, model.trainable_variables)
        g, _ = tf.clip_by_global_norm(g, 0.5)
        opt.apply_gradients(zip(g, model.trainable_variables))
        return loss
    for ep in range(epochs):
        order = balanced_indices(A, n_act)
        tot, nb = 0.0, 0
        for s in range(0, len(order), batch):
            bi = order[s:s+batch]
            tot += float(step(tf.constant(X[bi]), tf.constant(A[bi]), tf.constant(ADV[bi]),
                              tf.constant(R[bi]), tf.constant(old_logp[bi]))); nb += 1
        hist.append({'epoch': ep+1, 'loss': tot/nb})
    return hist


In [ ]:
# ── Cell 6: variants ──
# Baseline configuration (identical to Phase 1's attention+PPO-clip):
#   window 10/20, embed_dim 64, heads 4, epochs 150/200, batch 64, entropy 0.01/0.10
VARIANTS = {
    # label:        (kind, window_c, window_p, embed, heads, epochs_c, epochs_p, seeds)
    'attn_A_base':   ('attention', 10, 20, 64, 4, 150, 200, [1,2,3]),
    'attn_B_longer': ('attention', 10, 20, 64, 4, 300, 400, [1,2,3]),
    'attn_C_wider':  ('attention', 10, 20, 128, 8, 150, 200, [1,2,3]),
    'attn_D_context':('attention', 20, 40, 64, 4, 150, 200, [1,2,3]),
    'attn_E_combo':  ('attention', 20, 40, 128, 8, 300, 400, [1,2,3]),
    # capacity control for the reviewer's confound question — 5 seeds, baseline config
    'mlp_matched':   ('mlp_matched', 10, 20, 64, 4, 150, 200, [1,2,3,4,5]),
}
ENT = {'compaction': 0.01, 'partition': 0.10}

runs = []
for label, (kind, wc, wp, embed, heads, epc, epp, seeds) in VARIANTS.items():
    for seed in seeds:
        for spec in ['compaction', 'partition']:
            tag = f'{spec}_{label}_seed{seed}'
            wpath = OUT/'weights'/f'{tag}.weights.h5'
            if wpath.exists():
                print(f'skip {tag}'); continue
            set_seeds(seed)
            if spec == 'compaction':
                na, nf, ws, epochs = 4, 8, wc, epc
                X, A, ADV, R = windows_for('compaction', ws), Ac, ADV_C, Gc
            else:
                na, nf, ws, epochs = 5, 14, wp, epp
                X, A, ADV, R = windows_for('partition', ws), Ap, ADV_P, custom
            if kind == 'attention':
                model = AttentivePPOModel(na, nf, ws, embed_dim=embed, num_heads=heads)
            else:
                h1, h2 = MATCHED_MLP_HIDDEN[(na, nf, ws)]
                model = MlpPPOModel(na, nf, ws, hidden1=h1, hidden2=h2)
            model(tf.zeros((1, ws, nf)))
            t0 = time.time()
            hist = train_ppoclip(model, X, A, ADV, R, na, epochs, ent=ENT[spec])
            wall = time.time()-t0
            model.save_weights(str(wpath))
            pd.DataFrame(hist).to_csv(OUT/'curves'/f'{tag}_curve.csv', index=False)
            man = {'tag': tag, 'label': label, 'kind': kind, 'seed': seed, 'specialist': spec,
                   'algorithm': 'offline_ppo_clip_balanced', 'window_size': ws,
                   'embed_dim': embed if kind=='attention' else None,
                   'num_heads': heads if kind=='attention' else None,
                   'epochs': epochs, 'batch': 64, 'lr': 3e-4, 'entropy_coef': ENT[spec],
                   'num_actions': na, 'num_features': nf,
                   'trainable_params': nparams(model), 'n_train_samples': int(len(X)),
                   'wall_clock_sec': round(wall,1), 'final_loss': hist[-1]['loss'],
                   'gpu': GPU_NAME, 'tf_version': tf.__version__,
                   'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')}
            json.dump(man, open(OUT/'weights'/f'{tag}_manifest.json','w'), indent=2)
            runs.append(man)
            print(f'  {tag}: {wall:.0f}s  params={nparams(model):,}  loss={hist[-1]["loss"]:.4f}', flush=True)

pd.DataFrame(runs).to_csv(OUT/'training_summary.csv', index=False)
print(f'\n{len(runs)} runs complete')


In [ ]:
# ── Cell 7: package ──
zp = '/kaggle/working/phase7_weights.zip' if Path('/kaggle').exists() else './phase7_weights.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(OUT.parent))
print('→', zp)
print('Unpack into: revision/phase7_attention/   (giving revision/phase7_attention/phase7_weights/...)')
